In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install modelscope open_clip_torch einops ftfy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1

In [ ]:
from modelscope.hub.snapshot_download import snapshot_download

model_dir = snapshot_download('damo/cv_ddcolor_image-colorization', cache_dir='./modelscope')
print("Model downloaded to:", model_dir)

2025-07-30 12:11:48,608 - modelscope - WARNING - Model revision not specified, use revision: v1.02
2025-07-30 12:11:49,099 - modelscope - INFO - Got 7 files, start to download ...


Processing 7 items:   0%|          | 0.00/7.00 [00:00<?, ?it/s]

Model downloaded to: ./modelscope/damo/cv_ddcolor_image-colorization


In [ ]:
# GitHub 레포 다운로드
!git clone https://github.com/piddnad/DDColor.git
%cd DDColor

Cloning into 'DDColor'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 260 (delta 62), reused 56 (delta 40), pack-reused 156 (from 1)
Receiving objects: 100% (260/260), 15.21 MiB | 16.44 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/DDColor


In [ ]:
# 1. DDColor 레포 위치로 이동
%cd /content/DDColor

# 2. 출력 폴더 생성 (없으면)
!mkdir -p /content/output_images

# 3. infer.py 실행
!python infer.py \
  --model_path /content/modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt \
  --input /content/drive/MyDrive/INHA_AI_Challenge/open/test/input_image \
  --output /content/output_images

/content/DDColor
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Output path: /content/output_images
100% 200/200 [01:37<00:00,  2.06it/s]


In [ ]:
import open_clip
import torch
import pandas as pd
import numpy as np
import os, zipfile
from tqdm import tqdm
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
clip_model.to(device).eval()

TEST_CSV = "/content/drive/MyDrive/INHA_AI_Challenge/open/test.csv"
test_df = pd.read_csv(TEST_CSV)

ids, vecs = [], []
for row in tqdm(test_df.itertuples(), total=len(test_df)):
    img_id = row.ID
    # output_images 폴더에 저장된 컬러 PNG 경로
    base = os.path.basename(row.input_img_path)
    png_name = base.replace(".jpg", ".png")
    img_path = f"/content/output_images/{png_name}"

    img = Image.open(img_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = clip_model.encode_image(img_tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        emb = emb.cpu().numpy().reshape(-1)

    ids.append(img_id)
    vecs.append(emb)

cols = [f"vec_{i}" for i in range(vecs[0].shape[0])]
embed_df = pd.DataFrame(vecs, columns=cols)
embed_df.insert(0, "ID", ids)
embed_df.to_csv("/content/embed_submission.csv", index=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
100%|██████████| 200/200 [00:07<00:00, 27.62it/s]


In [ ]:
ZIP_PATH = "/content/DDColor_submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w") as z:
    # 컬러 PNG들
    for fname in sorted(os.listdir("/content/output_images")):
        z.write(f"/content/output_images/{fname}", arcname=fname)
    # 임베딩 CSV
    z.write("/content/embed_submission.csv", arcname="embed_submission.csv")

print("✅ embed_submission.csv 및 submission.zip 생성 완료:", ZIP_PATH)

✅ embed_submission.csv 및 submission.zip 생성 완료: /content/DDColor_submission.zip
